In [0]:
# print(bootstrap_servers)
# print(kafka_options["kafka.sasl.jaas.config"][:60] + "...")  

In [0]:
!pip install kafka-python

In [0]:
dbutils.library.restartPython()

In [0]:
#Checking if setup for eventhub is successful
# listen_conn_str = "Endpoint=sb://retailrocket-events.servicebus.windows.net/;SharedAccessKeyName=consumer-listen-policy;SharedAccessKey=laNoBrLtqps3F9p74ftrIrWDnnYHMWW2a+AEhK0s1dY="
# eventhub_name = "retailrocket-events"
# bootstrap_servers = "retailrocket-streaming-ns.servicebus.windows.net:9093"  # your actual namespace

# print("Setup complete")
# print(bootstrap_servers)

In [0]:
# from kafka.admin import KafkaAdminClient

# try:
#     admin = KafkaAdminClient(
#         bootstrap_servers="retailrocket-events.servicebus.windows.net:9093",
#         security_protocol="SASL_SSL",
#         sasl_mechanism="PLAIN",
#         sasl_plain_username="$ConnectionString",
#         sasl_plain_password=listen_conn_str,
#     )
#     print("Connected successfully:", admin.list_topics())
# except Exception as e:
#     print("Connection failed:", e)

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StringType, LongType

# Retrieve the Listen connection string
listen_conn_str = "Endpoint=sb://retailrocket-events.servicebus.windows.net/;SharedAccessKeyName=consumer-listen-policy;SharedAccessKey=laNoBrLtqps3F9p74ftrIrWDnnYHMWW2a+AEhK0s1dY="
eventhub_name = "retailrocket-events"

# Build the Kafka-compatible connection config (Event Hubs speaks Kafka protocol on Standard tier)
bootstrap_servers = "retailrocket-events.servicebus.windows.net:9093"
kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": eventhub_name,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{listen_conn_str}";',
    "startingOffsets": "earliest",
}

raw_stream = (spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

# Kafka source gives you raw binary 'value' column — this is your event JSON, needs parsing
event_schema = StructType() \
    .add("timestamp", LongType()) \
    .add("visitorid", StringType()) \
    .add("event", StringType()) \
    .add("itemid", StringType()) \
    .add("transactionid", StringType())

parsed_stream = (raw_stream
    .select(col("value").cast("string").alias("json_value"))
    .withColumn("data", from_json(col("json_value"), event_schema))
    .select("data.*")
    .withColumn("bronze_ingestion_time", current_timestamp())
)

(parsed_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/bronze_events/")
    .trigger(processingTime="10 seconds")
    .toTable("retrailrocket.bronze.events_stream")
)

In [0]:
from pyspark.sql.functions import col

# --- Category tree (static hierarchy, small file) ---
category_tree_df = spark.read.option("header", True).csv(
    "/Volumes/retrailrocket/landing/raw/Streaming/category_tree.csv"
)

category_tree_df.write.format("delta").mode("overwrite").saveAsTable(
    "retrailrocket.bronze.category_tree"
)

# --- Item properties (needs cleaning — recall the hashed 'n' prefix on numeric values) ---
item_props_part1 = spark.read.option("header", True).csv(
    "/Volumes/retrailrocket/landing/raw/Streaming/item_properties_part1.csv"
)
item_props_part2 = spark.read.option("header", True).csv(
    "/Volumes/retrailrocket/landing/raw/Streaming/item_properties_part2.csv"
)

item_props_raw = item_props_part1.union(item_props_part2)

item_props_raw.write.format("delta").mode("overwrite").saveAsTable(
    "retrailrocket.bronze.item_properties_raw"
)

# display(category_tree_df)
# display(item_props_raw.limit(20))

In [0]:
%sql
select count(*) from retrailrocket.bronze.events_stream